In [28]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [29]:
class Mystate(TypedDict):
    user_input: str
    result: str

In [30]:
def mainagent_node(state: Mystate):
    query = state["user_input"]
    # llm agent 로  query 실행

    return {"result": f"에이전트의 결과는 xxx입니다."}

In [31]:
graph_builder = StateGraph(Mystate)

graph_builder.add_node("mainagent", mainagent_node)

graph_builder.add_edge(START, "mainagent")

graph_builder.add_edge("mainagent", END)

In [32]:
app = graph_builder.compile()

result = app.invoke({"user_input": "안녕하세요"})
result

{'user_input': '안녕하세요', 'result': '에이전트의 결과는 xxx입니다.'}

In [ ]:
from dotenv import load_dotenv

load_dotenv()

True

In [34]:
from typing import TypedDict
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain_google_genai import ChatGoogleGenerativeAI

In [35]:
class BookState(TypedDict):  # 랭그래프 사용하는  state
    topic: str
    title: str


# structured out 스키마
class BookTitleOutput(BaseModel):
    """책 제목 생성 결과"""

    title: str = Field(description="사용자의 책 주제를 바탕으로 생성된 책 제목 1개")

In [36]:
fallback_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [37]:
# fallback middleware
@wrap_model_call
def fallback_model_middleware(request: ModelRequest, handler) -> ModelResponse:
    try:
        result = handler(request)
        return result

    except Exception as e:
        request.model = fallback_model

        return handler(request)

In [38]:
title_agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    system_prompt="""
    당신은 세계에서 제일 유명한 출판사의 메인 기획자입니다.
    사용자가 책의 주제를 입력하면 아래 조건을 만족하는 책의 제목을 1개만 만들어주세요.

        조건 :
        - 짧고 명확하게 작성
        - 너무 과장된 표현은 피할것
        - 제목만 출력할것
        - 베스트셀러가 될 제목으로만 만들것
        - 독자층의 취향을 고려해서 제목을 만들것
    """,
    response_format=BookTitleOutput,
    middleware=[fallback_model_middleware],
)


def title_agent_node(state: BookState) -> BookState:
    topic = state["topic"]

    result = title_agent.invoke(
        {"messages": [{"role": "user", "content": f"책의 주제 : {topic}"}]}
    )

    structrued_output: BookTitleOutput = result[
        "structured_response"
    ]  # {title:'책제목'}

    return {"title": structrued_output.title}

In [39]:
graph_builder = StateGraph(BookState)

graph_builder.add_node("title_agent_node", title_agent_node)

graph_builder.add_edge(START, "title_agent_node")
graph_builder.add_edge("title_agent_node", END)

agent = graph_builder.compile()

result = agent.invoke({"topic": "초등학생을 위한 RAG 에이전트 개발"})

print(result)

{'topic': '초등학생을 위한 RAG 에이전트 개발', 'title': '아이도 이해하는 RAG 에이전트'}
